# NASWOT при итеративном расширении модели на MNIST

Один и тот же набор из 10 мини-батчей оценивается до расширения и после каждого шага. Обучение не выполняется.


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *


In [ ]:
N = 5
hyperparams = {
    "batch_size": 256,
    "expansion_thresholds": {"fc_output": 0.4},
    "max_new_edges_per_expansion": 2000,
}


In [ ]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc_output = nn.Linear(input_size, 10)


    def forward(self, x):
        x = self.fc_output(x)
        return x


In [ ]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)

model.fc_output.dropout.p = 0.3

## Оценка NASWOT

Знак входа в ReLU/LeakyReLU определяет шаблон активации. Исходная линейная модель не имеет таких активаций, поэтому её оценка равна `-inf`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import islice
from ssonn.train.train import expand_model

@torch.inference_mode()
def naswot(model: nn.Module, x: torch.Tensor):
    K = torch.zeros((len(x), len(x)), dtype=torch.float32, device=x.device)
    n_activations = 0

    def count_patterns(_, inputs):
        nonlocal n_activations
        A = (inputs[0] > 0).flatten(1).float()
        K.add_(A @ A.T + (1 - A) @ (1 - A).T)
        n_activations += A.shape[1]

    hooks = [m.register_forward_pre_hook(count_patterns)
             for m in model.modules() if isinstance(m, (nn.ReLU, nn.LeakyReLU))]
    try:
        model.eval()(x)
    finally:
        for hook in hooks:
            hook.remove()

    _, logabsdet = torch.linalg.slogdet(K.cpu().double())
    return logabsdet.item(), n_activations


## Расширение и результаты


In [ ]:
eval_batches = [x.to(device) for x, _ in islice(val_loader, 10)]
edge_finder = EdgeFinder(
    metric=AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    dataloader=finder_loader,
    device=device,
    aggregation_mode="mean",
    max_to_replace=hyperparams["max_new_edges_per_expansion"],
)

records = []
new_edges = 0
for step in range(N + 1):
    for batch, x in enumerate(eval_batches):
        score, n_activations = naswot(model, x)
        records.append(dict(step=step, batch=batch, score=score,
                            n_activations=n_activations, new_edges=new_edges))
    if step < N:
        expanded, new_edges = expand_model(model, edge_finder, hyperparams["expansion_thresholds"])
        if not expanded:
            raise RuntimeError(f"Расширение на шаге {step + 1} не выбрало ни одной связи")

df = pd.DataFrame(records)
assert len(df) == 10 * (N + 1)
df


## Распределение оценок по шагам


In [ ]:
finite = df[np.isfinite(df["score"])]
steps = sorted(finite["step"].unique())
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([finite.loc[finite.step == step, "score"] for step in steps],
           positions=steps, widths=0.5)
rng = np.random.default_rng(SEED)
for step in steps:
    scores = finite.loc[finite.step == step, "score"]
    ax.scatter(step + rng.uniform(-0.12, 0.12, len(scores)), scores,
               alpha=0.6, s=18)
ax.set(xlabel="Число расширений", ylabel="NASWOT logdet", xticks=steps)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
